### **STEP 1 — Imports**

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

### **STEP 2 — Device**

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
# Add this right after device check
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# Test GPU works
if torch.cuda.is_available():
    test_tensor = torch.randn(3, 3).cuda()
    print("GPU test passed!")

CUDA available: True
CUDA device: NVIDIA GeForce GTX 1660 Ti
GPU test passed!


STEP 3 — Dataset

In [5]:
class SpectrogramDataset(Dataset):
    def __init__(self, root_dir):
        self.files = []
        self.labels = []
        
        for label, cls in enumerate(["real", "fake"]):
            folder = os.path.join(root_dir, cls)
            file_list = sorted(os.listdir(folder))  # Keeps same order as LightGBM
            
            for f in file_list:
                if f.endswith('.npy'):
                    self.files.append(os.path.join(folder, f))
                    self.labels.append(label)
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        spec = np.load(self.files[idx])
        spec = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return spec, label

STEP 4 — DataLoaders

In [ ]:
BASE = r"D:\Deepfake_Audio_Project"
SPECTRO = os.path.join(BASE, "features", "spectrograms")

train_loader = DataLoader(
    SpectrogramDataset(os.path.join(SPECTRO,"train")),
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

dev_loader = DataLoader(
    SpectrogramDataset(os.path.join(SPECTRO,"dev")),
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False

)

eval_loader = DataLoader(
    SpectrogramDataset(os.path.join(SPECTRO,"eval")),
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False

)

print("Data loaded")


Data loaded


STEP 5 — Improved CNN Architecture

In [7]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
            
            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
            
            # Block 3
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),
            
            # Block 4
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),
            
            nn.AdaptiveAvgPool2d((4, 4))
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNNModel().to(device)
print("\n✅ Model created")


✅ Model created


STEP 6 — Model Setup

In [ ]:
# Calculate class distribution
train_real = len(os.listdir(os.path.join(SPECTRO, "train", "real")))
train_fake = len(os.listdir(os.path.join(SPECTRO, "train", "fake")))
total = train_real + train_fake

# Calculate class weights (inverse frequency)
class_weights = torch.tensor([1.0, 3.0], dtype=torch.float32).to(device)

print("\n" + "="*60)
print("CLASS IMBALANCE HANDLING")
print("="*60)
print(f"Training data:")
print(f"  Real: {train_real} ({train_real/total*100:.1f}%)")
print(f"  Fake: {train_fake} ({train_fake/total*100:.1f}%)")
print(f"  Ratio: {train_fake/train_real:.2f}:1 (Fake:Real)")
print(f"\nClass weights applied:")

print("="*60)

# Weighted loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=5e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)


CLASS IMBALANCE HANDLING
Training data:
  Real: 2580 (10.2%)
  Fake: 22800 (89.8%)
  Ratio: 8.84:1 (Fake:Real)

Class weights applied:


STEP 7 — Training

In [11]:
EPOCHS = 20
best_val_loss = float('inf')
patience_counter = 0
patience_limit = 4
MODEL_PATH = os.path.join(BASE, "models")
os.makedirs(MODEL_PATH, exist_ok=True)

print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

for epoch in range(EPOCHS):
    # Training phase
    model.train()
    total_loss = 0
    for X, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)
    
    # Validation phase
    model.eval()
    val_loss = 0
    correct = 0
    total_samples = 0
    with torch.no_grad():
        for X, y in dev_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total_samples += y.size(0)
    avg_val_loss = val_loss / len(dev_loader)
    dev_acc = correct / total_samples
    
    print(f"Epoch {epoch+1:2d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {dev_acc:.4f}")
    
    # Learning rate scheduling
    scheduler.step(avg_val_loss)
    
    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(MODEL_PATH, "cnn_best.pth"))
        print(f"  ✅ Best model saved (Val Loss: {avg_val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"  ⚠️  No improvement ({patience_counter}/{patience_limit})")
    if patience_counter >= patience_limit:
        print(f"\n⏹️  Early stopping triggered at epoch {epoch+1}")
        break

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)
print("\n📂 Loading best model for evaluation...")
model.load_state_dict(torch.load(os.path.join(MODEL_PATH, "cnn_best.pth")))


STARTING TRAINING


Epoch 1/20: 100%|██████████| 794/794 [03:59<00:00,  3.32it/s]


Epoch  1 | Train Loss: 0.1143 | Val Loss: 0.1232 | Val Acc: 0.9334
  ✅ Best model saved (Val Loss: 0.1232)


Epoch 2/20: 100%|██████████| 794/794 [00:26<00:00, 30.14it/s]


Epoch  2 | Train Loss: 0.0328 | Val Loss: 0.0172 | Val Acc: 0.9950
  ✅ Best model saved (Val Loss: 0.0172)


...

Epoch 10/20: 100%|██████████| 794/794 [00:26<00:00, 29.51it/s]


Epoch 10 | Train Loss: 0.0046 | Val Loss: 0.0129 | Val Acc: 0.9957
  ⚠️  No improvement (3/4)


Epoch 11/20: 100%|██████████| 794/794 [00:25<00:00, 30.67it/s]


Epoch 11 | Train Loss: 0.0027 | Val Loss: 0.0154 | Val Acc: 0.9959
  ⚠️  No improvement (4/4)

⏹️  Early stopping triggered at epoch 11

TRAINING COMPLETED

📂 Loading best model for evaluation...


<All keys matched successfully>

In [15]:
# ===== QUICK EVAL CHECK (Before saving anything) =====
print("\n" + "="*60)
print("CHECKING EVAL PERFORMANCE (QUICK TEST)")
print("="*60)

model.eval()
eval_preds = []
eval_true = []

with torch.no_grad():
    for X, y in eval_loader:
        X = X.to(device)
        outputs = model(X)

        probs = torch.softmax(outputs, dim=1)[:, 1]

        threshold = 0.6
        preds = (probs > threshold).long()

        eval_preds.extend(preds.cpu().numpy())
        eval_true.extend(y.numpy())

eval_preds = np.array(eval_preds)
eval_true = np.array(eval_true)

from sklearn.metrics import classification_report, confusion_matrix

print("\n📊 EVAL Classification Report:")
print(classification_report(eval_true, eval_preds, target_names=['Real', 'Fake']))

print("\n📊 EVAL Confusion Matrix:")
cm = confusion_matrix(eval_true, eval_preds)
print(cm)

tn, fp, fn, tp = cm.ravel()
real_recall = tn / (tn + fp) if (tn + fp) > 0 else 0
fake_recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\n🎯 EVAL Per-Class Recall:")
print(f"Real recall: {real_recall:.4f} (needs to be > 0.50)")
print(f"Fake recall: {fake_recall:.4f} (needs to be > 0.90)")

if real_recall < 0.50:
    print("\n❌ MODEL FAILED: Not detecting real samples properly!")
    print("   Still overfitting to fake class.")
else:
    print("\n✅ MODEL LOOKS GOOD: Detecting both classes!")


CHECKING EVAL PERFORMANCE (QUICK TEST)

📊 EVAL Classification Report:
              precision    recall  f1-score   support

        Real       0.48      0.97      0.64      7355
        Fake       1.00      0.88      0.93     63882

    accuracy                           0.89     71237
   macro avg       0.74      0.93      0.79     71237
weighted avg       0.94      0.89      0.90     71237


📊 EVAL Confusion Matrix:
[[ 7155   200]
 [ 7835 56047]]

🎯 EVAL Per-Class Recall:
Real recall: 0.9728 (needs to be > 0.50)
Fake recall: 0.8774 (needs to be > 0.90)

✅ MODEL LOOKS GOOD: Detecting both classes!


STEP 8 — Save Model

In [16]:
MODEL_PATH = os.path.join(BASE,"models")
os.makedirs(MODEL_PATH, exist_ok=True)

torch.save(model.state_dict(), os.path.join(MODEL_PATH,"cnn_model.pth"))

print("CNN saved")


CNN saved


STEP 9 — Save DEV probabilities

In [17]:
dev_probs=[]
dev_labels=[]

model.eval()
with torch.no_grad():
    for X,y in dev_loader:
        X=X.to(device)
        probs=torch.softmax(model(X),dim=1)[:,1]
        dev_probs.extend(probs.cpu().numpy())
        dev_labels.extend(y.numpy())

np.save(os.path.join(MODEL_PATH,"cnn_dev_probs.npy"),np.array(dev_probs))
np.save(os.path.join(MODEL_PATH,"dev_labels_cnn.npy"),np.array(dev_labels))

print("Saved CNN DEV probabilities")


Saved CNN DEV probabilities


STEP 10 — Save EVAL probabilities

In [18]:
eval_probs=[]
eval_labels=[]

model.eval()
with torch.no_grad():
    for X,y in eval_loader:
        X=X.to(device)
        probs=torch.softmax(model(X),dim=1)[:,1]
        eval_probs.extend(probs.cpu().numpy())
        eval_labels.extend(y.numpy())

np.save(os.path.join(MODEL_PATH,"cnn_eval_probs.npy"),np.array(eval_probs))
np.save(os.path.join(MODEL_PATH,"eval_labels_cnn.npy"),np.array(eval_labels))

print("Saved CNN EVAL probabilities")


Saved CNN EVAL probabilities
